In [1]:
import os, time, json, csv, re, signal, threading, shutil, stat
import sys, subprocess
from pathlib import Path
from collections import deque

import numpy as np
import pandas as pd
import ipywidgets as W
from IPython.display import display, clear_output


# ---------------- Paths ----------------

NB_DIR = Path.cwd()

INPUT_DIR  = Path("../io/Input")
OUTPUT_DIR = Path("../io/Output")
TEST_DIR   = Path("../io/Test")

CORE_SCRIPT = Path("../system/engine_runtime.py")

TEST_X_PATH = TEST_DIR / "engine_total_X.npy"
TEST_CSV    = TEST_DIR / "test_eval.csv"

LIVE_JSON   = OUTPUT_DIR / "latest.json"
TEST_Y_PATH = TEST_DIR / "engine_total_benchmark_y.npy"


# ---------------- Helpers ----------------

def fmt_hms(sec: int) -> str:
    sec = int(max(0, sec))
    h = sec // 3600
    m = (sec % 3600) // 60
    s = sec % 60
    return f"{h:02d}:{m:02d}:{s:02d}"


def count_csv_rows(path: Path) -> int:
    full = NB_DIR / path

    if not full.exists():
        return 0

    try:
        with full.open("r", encoding="utf-8", errors="ignore") as f:
            return max(0, sum(1 for _ in f) - 1)
    except Exception:
        return 0


def count_total_from_X(path: Path) -> int:
    full = NB_DIR / path

    if not full.exists():
        return 0

    try:
        X = np.load(full, allow_pickle=True)

        total = 0

        for seq in X:
            arr = np.asarray(seq)

            if arr.ndim == 2:
                total += arr.shape[0]
            else:
                total += 1

        return total

    except Exception:
        return 0

# ---------------- Process Management ----------------
class ProcGroup:

    def __init__(self):
        self.core = None

    def start_core(self, test_mode: bool):

        full_script = NB_DIR / CORE_SCRIPT

        if not full_script.exists():
            return False, f"Missing: {full_script}"

        try:
            env = os.environ.copy()
            env["ENGINE_TEST_MODE"] = "1" if test_mode else "0"

            self.core = subprocess.Popen(
                [sys.executable, str(full_script)],
                cwd=str(NB_DIR),
                env=env,
                creationflags=subprocess.CREATE_NEW_PROCESS_GROUP,
            )

            return True, ""

        except Exception as e:
            return False, str(e)

    def stop_all(self):

        if self.core is not None:
            try:
                subprocess.run(
                    ["taskkill", "/F", "/T", "/PID", str(self.core.pid)],
                    stdout=subprocess.DEVNULL,
                    stderr=subprocess.DEVNULL,
                )
            except Exception:
                pass

        self.core = None
PROC = ProcGroup()

# ---------------- UI Visibility Control ----------------

def update_ui_visibility():

    if mode_toggle.value == "live":

        progress.layout.display = "none"
        rows_lbl.layout.display = "none"

        last_html.layout.display = None

        csv_box.layout.display = "none"
        csv_box.clear_output()

        show_csv_btn.layout.display    = "none"
        refresh_csv_btn.layout.display = "none"
        hide_csv_btn.layout.display   = "none"
        show_wrong_btn.layout.display = "none"

    else:

        progress.layout.display = None
        rows_lbl.layout.display = None

        last_html.layout.display = "none"

        show_csv_btn.layout.display    = None
        refresh_csv_btn.layout.display = None
        hide_csv_btn.layout.display   = None
        show_wrong_btn.layout.display = None

# ---------------- UI Components ----------------

last_html = W.HTML(
    """
    <div style="
        padding:20px;
        border-radius:12px;
        background:#f5f7fa;
        box-shadow:0 4px 12px rgba(0,0,0,0.08);
        text-align:center;
    ">
        <div style='font-size:14px; color:#666;'>ENGINE STATE</div>
        <div style='font-size:36px; font-weight:800; color:#111;'>—</div>
    </div>
    """
)


system_toggle = W.ToggleButtons(
    options=[("OFF", "off"), ("ON", "on")],
    value="off",
    description="System",
)


mode_toggle = W.ToggleButtons(
    options=[("TEST", "test"), ("LIVE", "live")],
    value="live",
    description="Mode",
)

status_html = W.HTML("<b>Status:</b> Stopped")

elapsed_lbl = W.Label("Elapsed: 00:00:00")

progress = W.FloatProgress(
    value=0.0,
    min=0.0,
    max=1.0,
    description="Progress",
    bar_style=''
)

rows_lbl = W.Label("")

err_out = W.Output(
    layout=W.Layout(
        border="1px solid #f00",
        display="none"
    )
)

display(
    W.VBox(
        [
            W.HBox(
                [
                    system_toggle,
                    mode_toggle,
                    status_html,
                    elapsed_lbl
                ]
            ),
            last_html,
            progress,
            rows_lbl,
            err_out
        ]
    )
)
# ---------------- CSV Viewer ----------------

def render_csv():

    csv_box.clear_output()

    full_csv = NB_DIR / TEST_CSV

    if not full_csv.exists():
        with csv_box:
            print("No CSV file found.")
        return

    try:
        df = pd.read_csv(full_csv)

        # ---- Efficient Cell Coloring (Only Predicted Label column) ----
        def highlight_row(row):
            pred = str(row["Predicted Label"]).strip()
            true = str(row["True Label"]).strip()

            styles = [""] * len(row)

            pred_col_index = row.index.get_loc("Predicted Label")

            if pred == true:
                styles[pred_col_index] = "background-color: #E8F5E9"
            else:
                styles[pred_col_index] = "background-color: #FFEBEE"

            return styles

        styled = df.style.apply(highlight_row, axis=1)

        with csv_box:
            display(styled)

    except Exception as e:
        with csv_box:
            print("CSV Load Error:", e)
# ---------------- CSV Viewer Controls ----------------

show_csv_btn = W.Button(
    description="Show CSV",
    button_style="primary"
)
refresh_csv_btn = W.Button(
    description="Refresh CSV"
)
hide_csv_btn = W.Button(
    description="Hide CSV",
    button_style="warning"
)
show_wrong_btn = W.Button(
    description="Show Only Wrong",
    button_style="danger"
)

csv_box = W.Output(
    layout=W.Layout(
        border="1px solid #ccc",
        max_height="400px",
        overflow="auto",
        display="none"
    )
)

csv_controls = W.HBox(
    [
        show_csv_btn,
        refresh_csv_btn,
        hide_csv_btn,
        show_wrong_btn
    ]
)

display(
    W.VBox(
        [
            csv_controls,
            csv_box
        ]
    )
)
update_ui_visibility()

# ---------------- Poller (Watcher Thread) ----------------

total_rows_cache = 0
stop_flag = False
start_time = None

def poller():

    global stop_flag
    global start_time
    global total_rows_cache

    while not stop_flag:

        if PROC.core is None:
            break

        exit_code = PROC.core.poll()

        # ---------------- Process still running ----------------

        if exit_code is None:

            status_html.value = (
                "<b>Status:</b> "
                "<span style='color:blue'>Running...</span>"
            )
            # elapsed time tracking

            if start_time is not None:

                elapsed = time.time() - start_time

                elapsed_lbl.value = (
                    f"Elapsed: {fmt_hms(elapsed)}"
                )

            # ---------------- LIVE MODE ----------------
            if mode_toggle.value == "live":

                try:

                    full_json = NB_DIR / LIVE_JSON

                    if full_json.exists():

                        with full_json.open(
                            "r",
                            encoding="utf-8"
                        ) as f:

                            data = json.load(f)

                        state = data.get("final", "—")
                        route = data.get("route", "")
                        window_data = data.get("window", [])

                        # ---------- COLOR LOGIC ----------

                        if "Engine Off (cold)" in state:
                            color = "#64B5F6"   # keep (perfect)

                        elif "Engine Off (warm)" in state:
                            color = "#4DD0E1"   # smooth blue→warm transition

                        elif "Engine Start" in state:
                            color = "#CE93D8"   # lighter transparent purple

                        elif "CriticalLoad" in state:
                            color = "#B71C1C"   # keep (perfect)

                        elif "HighLoad" in state:
                            color = "#EF6C00"   # strong but not red

                        elif "NormalLoad" in state:
                            color = "#F9A825"   # light warm yellow

                        elif "Frozen Sensor" in state:
                            color = "#BDBDBD"

                        elif "Uncalibrated" in state:
                            color = "#424242"

                        elif "Error Value" in state:
                            color = "#1A1A1A"

                        elif "Unknown" in state:
                            color = "#D32F2F"

                        else:
                            color = "#212121"
                        context_html = ""

                        if window_data:
                           rows_html = ""
                           for idx, row in enumerate(reversed(window_data)):
                               t, p, r, v = row

                               if idx == 0:
                                   row_style = 'style="background:#E3F2FD;"'
                               else:
                                   row_style = ""

                               rows_html += f"""
                               <tr {row_style}>
                                   <td style="text-align:left;">{-idx}</td>
                                   <td>{t:.6f}</td>
                                   <td>{p:.6f}</td>
                                   <td>{r:.6f}</td>
                                   <td>{v:.6f}</td>
                               </tr>
                               """

                           context_html = f"""
                           <div style="
                               margin-top:14px;
                               border-radius:10px;
                               background:white;
                               box-shadow: inset 0 0 0 1px #ddd;
                               overflow:hidden;
                               padding:8px;
                           ">
                               <div style="
                                   font-size:12px;
                                   font-weight:600;
                                   margin-bottom:6px;
                               ">
                                   Context Window (4s)
                               </div>

                               <table style="
                                   width:100%;
                                   font-family:monospace;
                                   font-size:12px;
                                   border-collapse:collapse;
                                   text-align:right;
                                   table-layout:fixed;
                               ">
                                   <thead>
                                       <tr style="background:#fafafa;">
                                           <th style="width:10%; text-align:left;">t</th>
                                           <th style="width:22%;">Temperature</th>
                                           <th style="width:22%;">Pressure</th>
                                           <th style="width:22%;">RPM</th>
                                           <th style="width:24%;">Vibration</th>
                                       </tr>
                                   </thead>
                                   <tbody>
                                       {rows_html}
                                   </tbody>
                               </table>
                           </div>
                           """
                        alert_section = ""

                        # -------- Alert Builder --------
                        if "Frozen Sensor" in state:
                            alert_section = """
                            <div style="
                                margin-top:14px;
                                padding:12px;
                                border-radius:8px;
                                background:#FFF3E0;
                                border:2px solid #E65100;
                                font-weight:600;
                            ">
                                ⚠️ Unknown [Frozen Sensor] — Sensor malfunction (no variation detected). Inspection required.
                            </div>
                            """

                        elif "Uncalibrated" in state:
                            alert_section = """
                            <div style="
                                margin-top:14px;
                                padding:12px;
                                border-radius:8px;
                                background:#FFF8E1;
                                border:2px solid #F9A825;
                                font-weight:600;
                            ">
                                🛠️ Unknown [Uncalibrated] — Uncalibrated sensor detected — Calibration required.
                            </div>
                            """

                        elif "Error Value" in state:
                            alert_section = """
                            <div style="
                                margin-top:14px;
                                padding:12px;
                                border-radius:8px;
                                background:#FFEBEE;
                                border:2px solid #B71C1C;
                                font-weight:600;
                            ">
                                ⚠️ Unknown [Output Error] — Invalid sensor output (non-numeric/corrupted). Immediate inspection required.
                            </div>
                            """

                        elif "CriticalLoad" in state:
                            alert_section = """
                            <div style="
                                margin-top:14px;
                                padding:12px;
                                border-radius:8px;
                                background:#FFEBEE;
                                border:2px solid #D32F2F;
                                font-weight:600;
                            ">
                                🚨 Critical load detected — Limit operation and service immediately.
                            </div>
                            """

                        # -------- Main Card --------
                        last_html.value = f"""
                        <div style="
                            padding:18px;
                            border-radius:14px;
                            background:#f5f7fa;
                            box-shadow:0 4px 16px rgba(0,0,0,0.08);
                            text-align:center;
                            min-height:140px;
                        ">
                            <div style='font-size:14px; color:#666;'>
                                ENGINE STATE
                            </div>

                            <div style='font-size:38px;
                                        font-weight:800;
                                        margin-top:8px;
                                        color:{color};'>
                                {state}
                            </div>

                            <div style='font-size:13px;
                                        color:#888;
                                        margin-top:6px;'>
                                {route}
                            </div>
                            {context_html}
                            {alert_section}

                        </div>
                        """

                except Exception:
                    pass

            # ---------------- TEST MODE ----------------

            if mode_toggle.value == "test":

                if total_rows_cache == 0:

                    total_rows_cache = count_total_from_X(
                        TEST_X_PATH
                    )

                done = count_csv_rows(TEST_CSV)

                # ---- Accuracy Calculation ----
                full_csv = NB_DIR / TEST_CSV

                accuracy_text = ""

                if full_csv.exists() and done > 0:
                    try:
                        df = pd.read_csv(full_csv)

                        correct = (
                            df["Predicted Label"].astype(str).str.strip() ==
                            df["True Label"].astype(str).str.strip()
                        ).sum()

                        accuracy = (correct / done) * 100

                        accuracy_text = f" | Accuracy: {accuracy:.4f}%"

                    except Exception:
                        accuracy_text = ""

                rows_lbl.value = (
                    f"Detected Rows in CSV: "
                    f"{done:,} / {total_rows_cache:,}"
                    f"{accuracy_text}"
                )
                if total_rows_cache > 0:

                    progress.value = min(
                        done / total_rows_cache,
                        1.0
                    )
                else:
                    progress.value = 0.0

        # ---------------- Process finished ----------------
        else:

            status_html.value = (
                "<b>Status:</b> "
                "<span style='color:green'>Finished</span>"
            )
            stop_flag = True
            break
        time.sleep(0.5)

# ---------------- Nuclear Shutdown ----------------

def nuclear_shutdown():
    global stop_flag
    stop_flag = True
    try:
        subprocess.run(
            [
                "taskkill",
                "/F",
                "/IM",
                "jupyter-nbconvert.exe",
                "/T"
            ],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
        subprocess.run(
            [
                "taskkill",
                "/F",
                "/IM",
                "python.exe",
                "/T"
            ],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )

        print(
            "Nuclear Shutdown: "
            "All background kernels purged."
        )

    except Exception as e:
        print(f"Shutdown warning: {e}")

# ---------------- System Toggle Handler ----------------

def on_system(change):

    global start_time
    global stop_flag
    global total_rows_cache

    val = change["new"]

    # ---------------- TURN ON ----------------

    if val == "on":
        total_rows_cache = 0
        stop_flag = True
        PROC.stop_all()
        time.sleep(1.0)
        err_out.clear_output()
        err_out.layout.display = "none"
        mode = mode_toggle.value
        ok, err = PROC.start_core(
            test_mode=(mode == "test")
        )
        if ok:
            start_time = time.time()

            stop_flag = False

            threading.Thread(
                target=poller,
                daemon=True
            ).start()
            target_path = (
                TEST_CSV
                if mode == "test"
                else LIVE_JSON
            )
            with err_out:

                print(f"MODE: {mode.upper()}")
                print(
                    f"Poller watching: "
                    f"{(NB_DIR / target_path).resolve()}"
                )
            err_out.layout.display = "block"
        else:
            with err_out:
                print(f"Failed: {err}")
            err_out.layout.display = "block"
            system_toggle.value = "off"
            
    # ---------------- TURN OFF ----------------
    else:
        stop_flag = True
        PROC.stop_all()
        time.sleep(0.5)

        status_html.value = (
            "<b>Status:</b> Stopped"
        )
        with err_out:
            print(
                "Ghost processes purged. System OFF."
            )
system_toggle.observe(
    on_system,
    names="value"
)

# ---------------- Mode Change Handler ----------------

def on_mode_change(change):
    update_ui_visibility()
    if system_toggle.value == "on":
        system_toggle.value = "off"
        time.sleep(0.5)
        system_toggle.value = "on"
mode_toggle.observe(
    on_mode_change,
    names="value"
)

# ---------------- CSV Button Handlers ----------------
def on_show_csv(b):
    if mode_toggle.value != "test":
        return
    csv_box.layout.display = "block"
    render_csv()


def on_refresh_csv(b):
    if mode_toggle.value != "test":
        return
    render_csv()


def on_hide_csv(b):
    csv_box.layout.display = "none"
    csv_box.clear_output()

def on_show_wrong(b):
    if mode_toggle.value != "test":
        return

    csv_box.layout.display = "block"
    csv_box.clear_output()

    full_csv = NB_DIR / TEST_CSV

    if not full_csv.exists():
        with csv_box:
            print("No CSV file found.")
        return

    try:
        df = pd.read_csv(full_csv)

        wrong_df = df[
            df["Predicted Label"].astype(str).str.strip() !=
            df["True Label"].astype(str).str.strip()
        ]

        with csv_box:
            display(wrong_df)

    except Exception as e:
        with csv_box:
            print("CSV Load Error:", e)


show_csv_btn.on_click(on_show_csv)
refresh_csv_btn.on_click(on_refresh_csv)
hide_csv_btn.on_click(on_hide_csv)
show_wrong_btn.on_click(on_show_wrong)